# M0 Batch Ingest — All 5 Fixture Filings

In [ ]:
import sys, json, time
from pathlib import Path
import pandas as pd
from IPython.display import display
sys.path.insert(0, str(Path("..").resolve()))

from ingestion.metadata_model import DocumentMetadata
from ingestion.downloader import FilingDownloader
from ingestion.sec_html_parser import SECHTMLParser
from ingestion.sec_chunker import SECChunker
from storage.writer import ChunkWriter

manifest = pd.read_csv("../fixtures/manifest.csv")
display(manifest[["slot","company_name","cik","filing_date"]])


## Download all filings (cached after first run)

In [ ]:
downloader = FilingDownloader(data_dir=Path("../data/raw"))
downloaded = []
for _, row in manifest.iterrows():
    meta = downloader.resolve_and_download(row)
    downloaded.append(meta)
    print(f"✓ {meta.company_name} — {meta.raw_html_path}")
    time.sleep(0.15)  # SEC rate limit: max ~6 req/s, stay conservative


## Parse, chunk, and store each filing

In [ ]:
parser = SECHTMLParser()
chunker = SECChunker()
writer = ChunkWriter()

summary = []
for meta in downloaded:
    raw_html = Path(meta.raw_html_path).read_text(encoding="utf-8", errors="replace")
    blocks = parser.parse(raw_html, meta)
    chunks = chunker.chunk_blocks(blocks, meta)
    written = writer.write_chunks(chunks, meta)
    summary.append({
        "company": meta.company_name,
        "blocks": len(blocks),
        "chunks": len(chunks),
        "stored": written,
    })
    print(f"✓ {meta.company_name}: {len(blocks)} blocks → {len(chunks)} chunks → {written} rows stored")

df_summary = pd.DataFrame(summary)
display(df_summary)


## Export batch manifest

In [ ]:
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(exist_ok=True)
df_summary.to_csv(OUTPUT_DIR / "m0_batch_summary.csv", index=False)
print("Exported output/m0_batch_summary.csv")


## M0 Gate Assertions

In [ ]:
for row in summary:
    assert row["blocks"] > 30, f"{row['company']}: expected >30 blocks, got {row['blocks']}"
    assert row["chunks"] > 30, f"{row['company']}: expected >30 chunks, got {row['chunks']}"
    assert row["stored"] == row["chunks"], f"{row['company']}: stored count mismatch"

total_chunks = sum(r["chunks"] for r in summary)
assert total_chunks > 200, f"Expected >200 total chunks across 5 filings, got {total_chunks}"
print(f"✓ All M0 gate assertions passed — {total_chunks} total chunks across {len(summary)} filings")
